# Notebook Overview: Spatial Operator Selection Ablation Study

This notebook implements the quantitative ablation study described in Section 4.2,
evaluating the UrbanTrace Copilot's automated spatial operator selection against four baselines
across 100 real-world NYC spatial integration scenarios.

## Pipeline Summary

**1. Build Dataset Registry** (`dataset_metadata_map.json`)
Scans all GeoJSON datasets and their metadata to extract geometry types and semantically
meaningful numeric columns, filtering out identifiers and boundary datasets.

**2. Inspect Registry Statistics**
Summarizes the corpus: dataset count, geometry type distribution, and target column counts.

**3. LLM-Assisted Ground Truth Annotation** (`ground_truth_llm_annotation.json`)
Uses GPT-4o-Mini via Portkey to classify each column by semantic class (`C_ext`, `C_int`,
`C_ord`) and assign a default aggregation operator — producing a preliminary annotation
for expert review. 

**4. Generate Evaluation Scenarios** (`scenarios_from_benchmark.json`)
Randomly pairs source dataset columns (from the human-curated ground truth (`ground_truth_curated.json`)) with target
boundary datasets to produce 100 distinct spatial mapping scenarios.

**5. Run Master Benchmark** (`benchmark_results_operator_selection.json`)
Evaluates all 5 conditions — Rule-Based, GPT-4o-Mini, Claude Sonnet, Claude Opus, and the
UrbanTrace Copilot — on every scenario, measuring **Geometric Validity** (correct mapping
operator) and **Semantic Validity** (correct aggregation operator). Results are compiled into
the accuracy table reported in the paper.


## Build Dataset Registry

Scans all GeoJSON files in `../data/geojson` and their corresponding metadata files in
`../data/metadata` to construct a unified dataset registry (`dataset_metadata_map.json`).

For each matched dataset, it extracts:
- **Geometry type** from metadata
- **Target columns**: numeric columns (Integer, Number, Float) that are *not* identifiers,
  coordinates, or administrative labels (filtered via an exclusion list)
- **Data type**: classified as `"boundary"` (e.g., Council Districts, NTAs) or `"data"`;
  boundary datasets get no target columns

Two datasets (`NYC_Issued_Licenses`, `NYPD_arrests_data`) are explicitly excluded from the
final output. The resulting registry is saved to `dataset_metadata_map.json`.

In [17]:
from pathlib import Path
import json

geojson_path = Path("./data/geojson")
metadata_path = Path("./data/metadata")

boundary_datasets = {
    "City_Council_Districs",
    "Community_Districs",
    "MODZCTA",
    "NTA_Neighborhood_Tabulation_Areas",
    "NYC_Bike_Routes"
}

datasets = sorted([p.stem for p in geojson_path.glob("*.geojson")])
metadata_files = {p.stem for p in metadata_path.glob("*")}

dataset_map = {}

matched = []
missing_metadata = []
extra_metadata = []

# helper: load metadata JSON-like structure if needed later
def extract_metadata_fields(meta_path):
    with open(meta_path, "r") as f:
        return json.load(f)

for name in datasets:
    metadata_name = f"{name}_metadata"

    if metadata_name not in metadata_files:
        missing_metadata.append(name)
        continue

    matched.append(name)

    # load metadata file (assumes json format; adjust if needed)
    meta_file = metadata_path / f"{metadata_name}.json"
    if not meta_file.exists():
        # fallback if extension unknown
        meta_file = next(metadata_path.glob(f"{metadata_name}.*"), None)

    if not meta_file:
        continue

    metadata = extract_metadata_fields(meta_file)

    # ----------------------------
    # geometry type
    # ----------------------------
    geometry_type = metadata.get("geometricType")

    # ----------------------------
    # extract numeric columns
    # ----------------------------
    target_columns = []

    for col in metadata.get("columns", []):
        structural_type = col.get("structural_type", "")

        if structural_type in [
            "http://schema.org/Integer",
            "http://schema.org/Number",
            "http://schema.org/Float"
        ]:
            target_columns.append(col.get("name"))

    # ----------------------------
    # classify dataset type
    # ----------------------------
    data_type = "boundary" if name in boundary_datasets else "data"
    
    # ----------------------------
    # extract numeric columns ONLY if not boundary
    # exclude columns from list
    # ----------------------------

    # ensure target_columns only contains true numeric signals, not IDs, coordinates, or administrative labels.
    exclude_columns_raw = [
        "ZIP Code Tabulation Area (ZCTA) 2020",
        "zipcode", "bin", "bbl", "latitude", "longitude", "xcoord", "ycoord",
        "cd", "council", "ct2010", "ct2020", "borocode", "schooldist", "policeprct",
        "OBJECTID", "Borough", "Latitude", "Longitude", "BoroCode", "Council Distrcit",
        "Postcode", "BoroCD", "Census Tract", "BCTCB2010", "BIN", "BBL", "DOITT_ID",
        "RequestID", "Yr", "M", "D", "HH", "MM", "SegmentID", "countid", "id", "status",
        "lon", "lat", "bor_subb", "ZIP CODE", "LATITUDE", "LONGITUDE", "COLLISION_ID", "Loc"
    ]
    
    # normalize once
    exclude_set = set(x.strip().lower() for x in exclude_columns_raw)


    #########
    
    target_columns = []

    if data_type != "boundary":
        for col in metadata.get("columns", []):
            col_name = col.get("name", "")
            structural_type = col.get("structural_type", "")
    
            # must be numeric
            is_numeric = structural_type in [
                "http://schema.org/Integer",
                "http://schema.org/Number",
                "http://schema.org/Float"
            ]
    
            # normalize for comparison
            col_name_norm = col_name.strip().lower()
    
            # exclude if in blacklist
            is_excluded = col_name_norm in exclude_set
    
            if is_numeric and not is_excluded:
                target_columns.append(col_name)
    
    # ----------------------------
    # enforce rule explicitly (safety)
    # ----------------------------
    if data_type == "boundary":
        target_columns = []
    
    dataset_map[name] = {
        "dataset": name,
        "metadata": metadata_name,
        "data_type": data_type,
        "geometry_type": geometry_type,
        "target_columns": target_columns
    }

# extra metadata check
dataset_set = set(datasets)
for m in metadata_files:
    if m.endswith("_metadata"):
        dataset_name = m.replace("_metadata", "")
        if dataset_name not in dataset_set:
            extra_metadata.append(m)

# ----------------------------
# DEBUG LOGS (not saved)
# ----------------------------
print("✅ Matched datasets:", len(matched))
print("❌ Missing metadata:", len(missing_metadata))
print("⚠️ Extra metadata files:", len(extra_metadata))

print("\nMissing metadata for:")
for m in missing_metadata:
    print(f" - {m}")

print("\nExtra metadata files (no dataset):")
for e in extra_metadata:
    print(f" - {e}")


# Remove these datsets
# datasets to exclude from final output
remove_datasets = {"NYC_Issued_Licenses", "NYPD_arrests_data"}

# filter them out of final map
dataset_map = {
    k: v for k, v in dataset_map.items()
    if k not in remove_datasets
}


# ----------------------------
# SAVE CLEAN REGISTRY
# ----------------------------
output_path = Path("dataset_metadata_map.json")

with open(output_path, "w") as f:
    json.dump(dataset_map, f, indent=2)

print(f"\nSaved {len(dataset_map)} datasets to {output_path}")

✅ Matched datasets: 27
❌ Missing metadata: 1
⚠️ Extra metadata files: 0

Missing metadata for:
 - nyc-zip-code-tabulation-areas-polygons

Extra metadata files (no dataset):

Saved 25 datasets to dataset_metadata_map.json


## Dataset Registry Statistics

Loads `dataset_metadata_map.json` and prints a summary of the registry contents:

- **Total dataset count**
- **Geometry type distribution** (e.g., MultiPolygon, Point, MultiLineString)
- **Target column counts**: total columns across all datasets (including duplicates)
  and the number of unique column names

In [18]:
from pathlib import Path
import json
from collections import Counter

# Load dataset map
file_path = Path("dataset_metadata_map.json")

with open(file_path, "r") as f:
    dataset_map = json.load(f)

# ----------------------------
# Basic counts
# ----------------------------
num_datasets = len(dataset_map)

# ----------------------------
# Geometry types
# ----------------------------
geometry_types = [
    v.get("geometry_type") for v in dataset_map.values()
]
geometry_counter = Counter(geometry_types)

# ----------------------------
# Target columns stats
# ----------------------------
all_target_columns = [
    col
    for v in dataset_map.values()
    for col in v.get("target_columns", [])
]

num_total_columns = len(all_target_columns)
num_unique_columns = len(set(all_target_columns))

# ----------------------------
# Print stats
# ----------------------------
print("📊 DATASET STATISTICS")
print("-" * 30)

print(f"Total datasets: {num_datasets}")

print("\n🗺 Geometry types:")
for gtype, count in geometry_counter.items():
    print(f"  {gtype}: {count}")

print("\n📈 Columns:")
print(f"  Total target columns (with duplicates): {num_total_columns}")
print(f"  Unique target columns: {num_unique_columns}")

📊 DATASET STATISTICS
------------------------------
Total datasets: 25

🗺 Geometry types:
  MultiPolygon: 17
  MultiLineString: 1
  Point: 7

📈 Columns:
  Total target columns (with duplicates): 215
  Unique target columns: 43


## LLM-Assisted Ground Truth Annotation

Uses GPT-4o-Mini (via Portkey AI Gateway) to semantically annotate every target column
in the dataset registry, producing a preliminary ground truth for manual curation.

For each column, the model receives the dataset name, geometry type, column statistics
(mean, stddev, distinct values), and sample values, then classifies it into:

- **Semantic class**: `C_ext` (extensive/count), `C_int` (intensive/rate), or `C_ord` (ordinal/index)
- **Default aggregation operator**: selected from a fixed registry
  (`SumZoning`, `WeightedMeanZoning`, `DensityZoning`, `MajorityZoning`, etc.)

Results are saved to `ground_truth_llm_annotation.json` and are intended as a
starting point for expert review, not as final ground truth.

In [ ]:
YOUR_PORTKEY_API_KEY = "YOUR_PORTKEY_API_KEY"
PORTKEY_BASE_URL = "YOUR_PORTKEY_BASE_URL"

In [22]:
import json
import os
import csv
import io
import time
from portkey_ai import Portkey
from tqdm.notebook import tqdm

# --- Configuration ---
MAP_FILE = "dataset_metadata_map.json"
METADATA_DIR = "./data/metadata"
OUTPUT_FILE = "ground_truth_llm_annotation.json"

# Initialize Portkey Client
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY", YOUR_PORTKEY_API_KEY) # Use your real key

client = Portkey(
    base_url=PORTKEY_BASE_URL,
    api_key=PORTKEY_API_KEY
)

# --- The Prompt Template ---
SYSTEM_PROMPT = """
You are an expert urban data scientist and GIS spatial analyst.
Your task is to analyze a dataset column and annotate its spatial semantic class and default aggregation operator.

Step 1: Determine the Semantic Class
Analyze the provided statistics and sample values using these strict rules:
- "C_ext" (Extensive - Count/Total): Typically has high num_distinct_values, larger means, and values that scale with the physical area (e.g., total population, number of crimes).
- "C_int" (Intensive - Rate/Density): Often decimals/floats, uses keywords like rate/avg/median/density. These do NOT scale with physical area (e.g., poverty rate, median income).
- "C_ord" (Categorical/Ordinal - Index): Typically has low num_distinct_values (often 1-10), integer classes, or index-like labels (e.g., zoning codes, vulnerability rank 1-5).

Step 2: Determine the Target Aggregation Operator
You must select the most mathematically appropriate default aggregation operator STRICTLY from the following system registry:
["SumZoning", "WeightedMeanZoning", "DensityZoning", "MajorityZoning", "MaxZoning", "MinZoning", "LengthWeightedZoning"]

Rules for Operator Selection:
- If C_ext -> Aggregation must preserve totals. Select: "SumZoning"
- If C_int -> Aggregation must avoid absurd accumulation. Select: "WeightedMeanZoning" (or "DensityZoning" if it is explicitly a spatial density).
- If C_ord -> Use discrete grouping. Select: "MajorityZoning" (or "MaxZoning"/"MinZoning" if it is a priority index).

Output STRICTLY in the following JSON format:
{
  "column_name": "<exact column name>",
  "inferred_meaning": "<A 1-2 sentence explanation of what this data represents in the real world>",
  "semantic_class": "<C_ext, C_int, or C_ord>",
  "default_aggregation": "<Exact operator name from the registry list>"
}
"""

def extract_sample_values(sample_csv_string, column_name, max_samples=5):
    if not sample_csv_string: return []
    try:
        reader = csv.DictReader(io.StringIO(sample_csv_string))
        samples = []
        for row in reader:
            if column_name in row and row[column_name].strip() != "":
                samples.append(row[column_name])
            if len(samples) >= max_samples: break
        return samples
    except Exception as e:
        return []

def get_column_metadata(columns_list, target_column_name):
    for col in columns_list:
        if col.get("name") == target_column_name: return col
    return {}

def annotate_column(dataset_name, dataset_info, column_name, col_meta, sample_values):
    stats_str = f"Mean: {col_meta.get('mean', 'N/A')}, StdDev: {col_meta.get('stddev', 'N/A')}, Distinct Values: {col_meta.get('num_distinct_values', 'N/A')}"
    
    user_prompt = f"""
    DATASET CONTEXT:
    - Dataset Name: {dataset_name}
    - Geometry Type: {dataset_info.get('geometry_type')}
    - Data Type: {dataset_info.get('data_type')}
    
    COLUMN TO ANALYZE:
    - Name: {column_name}
    - Structural Type: {col_meta.get('structural_type', 'Unknown')}
    - Statistics: {stats_str}
    - Sample Values: {sample_values}
    """
    
    try:
        response = client.chat.completions.create(
            model="@gpt-4o-mini/gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" },
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"❌ Error processing {dataset_name} - {column_name}: {e}")
        return None

# --- Main Execution Loop ---
def generate_ground_truth():
    with open(MAP_FILE, 'r') as f:
        dataset_map = json.load(f)
        
    ground_truth = {}
    total_columns = sum(len(info.get('target_columns', [])) for info in dataset_map.values())
    print(f"🚀 Launching batch annotation for {total_columns} columns across {len(dataset_map)} datasets...")
    
    with tqdm(total=total_columns, desc="Processing") as pbar:
        for dataset_name, info in dataset_map.items():
            target_cols = info.get('target_columns', [])
            if not target_cols:
                continue
                
            if dataset_name not in ground_truth:
                ground_truth[dataset_name] = {}
                
            meta_filename = info.get('metadata') + ".json" 
            meta_path = os.path.join(METADATA_DIR, meta_filename)
            
            if not os.path.exists(meta_path):
                print(f"⚠️ Warning: Metadata file not found -> {meta_path}")
                pbar.update(len(target_cols))
                continue
                
            with open(meta_path, 'r') as mf:
                metadata = json.load(mf)
                
            columns_list = metadata.get('columns', [])
            sample_csv_string = metadata.get('sample', '')
            
            for col_name in target_cols:
                col_meta = get_column_metadata(columns_list, col_name)
                sample_values = extract_sample_values(sample_csv_string, col_name)
                
                # Call Portkey
                annotation = annotate_column(dataset_name, info, col_name, col_meta, sample_values)
                if annotation:
                    ground_truth[dataset_name][col_name] = annotation
                
                # Sleep briefly to avoid API rate limits
                time.sleep(0.5) 
                pbar.update(1)
                
    with open(OUTPUT_FILE, 'w') as out_f:
        json.dump(ground_truth, out_f, indent=2)
        
    print(f"\n✅ Semantic annotation complete! Gold standard saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    generate_ground_truth()

🚀 Launching batch annotation for 215 columns across 25 datasets...


Processing:   0%|          | 0/215 [00:00<?, ?it/s]


✅ Semantic annotation complete! Gold standard saved to ground_truth_final.json


## Ground Truth Manual Annotation

Review and manually curate the contents of `ground_truth_llm_annotation.json` to produce the finalized ground truth file, `ground_truth_curated.json`.

## Generate Random Spatial Integration Scenarios

Constructs 100 random spatial mapping scenarios by pairing source dataset columns
with target boundary datasets, drawing from the human-curated ground truth
(`ground_truth_curated.json`).

Each scenario records:
- **Source**: dataset name, geometry type, column name, and ground truth semantic
  class + aggregation operator
- **Target**: boundary dataset name and geometry type

Scenarios are saved to `scenarios_from_benchmark.json` and serve as the evaluation
corpus for benchmarking the ablation conditions.

In [40]:
import json
import random
import os
from portkey_ai import Portkey
from tqdm.notebook import tqdm
import pandas as pd

# --- Step 1: Scenario Generator ---
def generate_scenarios(GROUND_TRUTH_FILE, MAP_FILE, SCENARIO_OUTPUT, n=5):
    # --- Load Data ---
    with open(MAP_FILE, 'r') as f:
        dataset_map = json.load(f)
    with open(GROUND_TRUTH_FILE, 'r') as f:
        ground_truth = json.load(f)

    
    """Generates N random (Source Dataset/Column -> Target Boundary) scenarios."""
    scenarios = []
    
    # 1. Get all valid source columns that we have ground truth for
    source_candidates = []
    for ds_name, cols in ground_truth.items():
        for col_name, gt_data in cols.items():
            source_candidates.append({
                "source_dataset": ds_name,
                "source_geom": dataset_map[ds_name]["geometry_type"],
                "column_name": col_name,
                "gt_semantic": gt_data["semantic_class"],
                "gt_aggregation": gt_data["default_aggregation"]
            })
            
    # 2. Get all valid target boundaries
    target_candidates = [
        ds_name for ds_name, info in dataset_map.items() 
        if info.get("data_type") == "boundary"
    ]
    
    for _ in range(n):
        src = random.choice(source_candidates)
        tgt_name = random.choice(target_candidates)
        tgt_geom = dataset_map[tgt_name]["geometry_type"]
        
        scenarios.append({
            "source": src,
            "target_dataset": tgt_name,
            "target_geom": tgt_geom
        })

    # ✅ Save to JSON before returning
    with open(SCENARIO_OUTPUT, "w") as f:
        json.dump(scenarios, f, indent=4)

    return scenarios

In [41]:
# --- Configuration ---
MAP_FILE = "dataset_metadata_map.json"
GROUND_TRUTH_FILE = "ground_truth_curated.json" # human-verified (and corrected) over LLM-generated ("ground_truth_llm_annotation.json")
NUM_SCENARIOS = 100 # How many random combinations to test
SCENARIO_OUTPUT = "scenarios_from_benchmark.json"
scenarios = generate_scenarios(GROUND_TRUTH_FILE, MAP_FILE, SCENARIO_OUTPUT, NUM_SCENARIOS) 
scenarios

[{'source': {'source_dataset': 'NYC_median_rent',
   'source_geom': 'MultiPolygon',
   'column_name': '2018',
   'gt_semantic': 'C_int',
   'gt_aggregation': 'WeightedMeanZoning'},
  'target_dataset': 'Community_Districs',
  'target_geom': 'MultiPolygon'},
 {'source': {'source_dataset': 'NYC_vehicle_collisions_crashes',
   'source_geom': 'Point',
   'column_name': 'NUMBER OF PEDESTRIANS INJURED',
   'gt_semantic': 'C_ext',
   'gt_aggregation': 'SumZoning'},
  'target_dataset': 'NYC_Bike_Routes',
  'target_geom': 'MultiLineString'},
 {'source': {'source_dataset': 'NYC_vehicle_collisions_crashes',
   'source_geom': 'Point',
   'column_name': 'NUMBER OF PEDESTRIANS KILLED',
   'gt_semantic': 'C_ext',
   'gt_aggregation': 'SumZoning'},
  'target_dataset': 'City_Council_Districs',
  'target_geom': 'MultiPolygon'},
 {'source': {'source_dataset': 'NYC_vehicle_collisions_crashes',
   'source_geom': 'Point',
   'column_name': 'NUMBER OF CYCLIST INJURED',
   'gt_semantic': 'C_ext',
   'gt_aggreg

## Master Benchmark Pipeline

Runs the full ablation study across all 5 conditions on the 100 generated scenarios,
comparing operator selection accuracy for both evaluation dimensions.

**Conditions evaluated:**
1. **Rule-Based (Legacy GIS)** — keyword matching on column name + random geometry guess
2. **Naive LLM (GPT-4o-Mini)** — column/dataset names only, no statistical context
3. **Naive LLM (Claude Sonnet)** — same minimal context, frontier model
4. **Naive LLM (Claude Opus)** — same minimal context, frontier model
5. **UrbanTrace Copilot** — live backend API call with full statistical profiling context

Each prediction is evaluated against ground truth on:
- **Geometric Validity**: whether the correct spatial mapping operator was selected
  (e.g., `CentroidZoning` for Points, `AreaWeightedZoning` for Polygons)
- **Semantic Validity**: whether the aggregation operator matches the curated ground truth

Per-scenario results are saved to `benchmark_results_operator_selection.json`,
and a summary accuracy table (%) is printed at the end.

In [62]:
import json
import pandas as pd
from tqdm.notebook import tqdm
from portkey_ai import Portkey
import os
import requests # Make sure to import this at the top of your notebook

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY", YOUR_PORTKEY_API_KEY)
client = Portkey(base_url=PORTKEY_BASE_URL, api_key=PORTKEY_API_KEY)

NUM_SCENARIOS = 100
MASTER_RESULTS_FILE = "benchmark_results_operator_selection.json"
scenarios_final = "scenarios_from_benchmark.json"

# Model Strings
SONNET_MODEL_STRING = "@bedrock/us.anthropic.claude-sonnet-4-20250514-v1:0"
OPUS_MODEL_STRING = "@vertexai/anthropic.claude-opus-4-6"

def run_copilot_api(scenario_context):
    """Hits your actual live local backend endpoint to get the system's recommendation."""
    url = "http://localhost:8000/api/v1/copilot/recommend-operators"
    
    # Format the payload exactly as your system expects it (adding .geojson)
    payload = {
        "target_zoning": {
            "dataset_name": f"{scenario_context['target_dataset']}.geojson",
            "geometry_type": scenario_context['target_geometry']
        },
        "source_variables": [
            {
                "dataset_name": f"{scenario_context['source_dataset']}.geojson",
                "column_name": scenario_context['source_column'],
                "original_geometry": scenario_context['source_geometry']
            }
        ]
    }
    
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        
        # Parse the JSON response array
        data = response.json()
        
        # Since we are only sending one source variable, we grab the first item in the array
        if data and len(data) > 0:
            result = data[0]
            return result.get("zoningMapping", "Error"), result.get("zoningAggregation", "Error")
        else:
            return "Error", "Error"
            
    except requests.exceptions.RequestException as e:
        print(f"API Request Failed for {scenario_context['source_column']}: {e}")
        return "Error", "Error"

# ==========================================
# 3. BASELINES & EVALUATION LOGIC
# ==========================================
def evaluate_mapping_operator(src_geom, predicted_map):
    """The Deterministic Rule Matrix for Geometry"""
    if "Point" in src_geom: return predicted_map == "CentroidZoning"
    elif "Line" in src_geom: return predicted_map == "LengthWeightedZoning"
    elif "Polygon" in src_geom: return predicted_map == "AreaWeightedZoning"
    return False

import random
def run_rule_based_baseline(scenario_context):
    """Guesses operators using rigid keyword matching (simulating legacy GIS)."""
    col_name = scenario_context["source_column"].lower()
    
    # Aggregation Guess
    if any(word in col_name for word in ["rate", "avg", "mean", "index", "density"]):
        agg = "WeightedMeanZoning"
    elif any(word in col_name for word in ["id", "code", "category", "type"]):
        agg = "MajorityZoning"
    else:
        agg = "SumZoning" # Default to sum
        
    # Mapping Guess
    # src_g = scenario_context["source_geometry"]
    map_op = random.choice([
        "CentroidZoning",
        "LengthWeightedZoning",
        "AreaWeightedZoning"
    ])
    return map_op, agg

def run_naive_llm(scenario_context):
    """Simulates a baseline LLM (No Atlas Profiler, No Backend Context)."""
    sys_prompt = """You are an automated spatial operator selection agent.
    Pick the best Mapping Operator from: ["CentroidZoning", "AreaWeightedZoning", "LengthWeightedZoning"]
    Pick the best Aggregation Operator from: ["SumZoning", "WeightedMeanZoning", "DensityZoning", "MajorityZoning", "MaxZoning", "MinZoning", "LengthWeightedZoning"]
    
    Output JSON ONLY: {"mapping_operator": "...", "aggregation_operator": "..."}"""
    
    user_prompt = f"""
    Task: Map data from Source to Target.
    Source Dataset: {scenario_context['source_dataset']} 
    Target Boundary: {scenario_context['target_dataset']}
    Column to Map: {scenario_context['source_column']}
    
    No statistical metadata available.
    """
    
    try:
        res = client.chat.completions.create(
            model="@gpt-4o-mini/gpt-4o-mini",
            messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        data = json.loads(res.choices[0].message.content)
        return data.get("mapping_operator", "Error"), data.get("aggregation_operator", "Error")
    except Exception:
        return "Error", "Error"

def run_naive_frontier_model(scenario_context, model_string):
    """Runs Naive LLM baseline with robust JSON extraction to bypass Markdown hallucination."""
    sys_prompt = """You are an automated spatial operator selection agent.
    Pick the best Mapping Operator from: ["CentroidZoning", "AreaWeightedZoning", "LengthWeightedZoning"]
    Pick the best Aggregation Operator from: ["SumZoning", "WeightedMeanZoning", "DensityZoning", "MajorityZoning", "MaxZoning", "MinZoning", "LengthWeightedZoning"]
    
    Output STRICTLY ONLY JSON. No markdown, no backticks, no conversational text. Format:
    {"mapping_operator": "...", "aggregation_operator": "..."}"""
    
    user_prompt = f"""
    Task: Map data from Source to Target.
    Source Dataset: {scenario_context['source_dataset']}
    Target Boundary: {scenario_context['target_dataset']}
    Column to Map: {scenario_context['source_column']}
    
    No statistical metadata available.
    """
    
    try:
        res = client.chat.completions.create(
            model=model_string,
            messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
            temperature=0.0
        )
        
        raw_content = res.choices[0].message.content.strip()
        
        # Robust JSON Extraction
        start_idx = raw_content.find('{')
        end_idx = raw_content.rfind('}')
        
        if start_idx != -1 and end_idx != -1:
            clean_json_str = raw_content[start_idx:end_idx+1]
            data = json.loads(clean_json_str)
            return data.get("mapping_operator", "Error"), data.get("aggregation_operator", "Error")
        else:
            return "Error", "Error"
            
    except Exception as e:
        print(f"API/Parsing Error for {model_string}: {e}")
        return "Error", "Error"

# ==========================================
# 4. MASTER BENCHMARK PIPELINE
# ==========================================
def run_master_benchmark():
    print(f"🚀 Initializing Master Benchmark across 5 conditions ({NUM_SCENARIOS} scenarios)...")

    
    with open(scenarios_final, 'r') as f:
        scenarios = json.load(f)
    # scenarios = generate_scenarios(NUM_SCENARIOS) 
    results = []
    
    for s in tqdm(scenarios, desc="Evaluating Scenarios"):
        # --- Context Extraction ---
        ctx = {
            "source_dataset": s["source"]["source_dataset"],
            "source_column": s["source"]["column_name"],
            "source_geometry": s["source"]["source_geom"],
            "target_dataset": s["target_dataset"],
            "target_geometry": s["target_geom"],
            "ground_truth_semantic": s["source"]["gt_semantic"],
            "ground_truth_aggregation": s["source"]["gt_aggregation"]
        }
        
        src_geom = ctx["source_geometry"]
        gt_agg = ctx["ground_truth_aggregation"]
        
        # --- Condition 1: Rule-Based (Legacy) ---
        rb_map, rb_agg = run_rule_based_baseline(ctx)
        rb_geom_val = evaluate_mapping_operator(src_geom, rb_map)
        rb_sem_val = (rb_agg == gt_agg)
        
        # --- Condition 2: Naive LLM (GPT-4o-Mini) ---
        mini_map, mini_agg = run_naive_llm(ctx)
        mini_geom_val = evaluate_mapping_operator(src_geom, mini_map)
        mini_sem_val = (mini_agg == gt_agg)
        
        # --- Condition 3: Naive LLM (Claude Sonnet) ---
        sonnet_map, sonnet_agg = run_naive_frontier_model(ctx, SONNET_MODEL_STRING)
        sonnet_geom_val = evaluate_mapping_operator(src_geom, sonnet_map)
        sonnet_sem_val = (sonnet_agg == gt_agg)
        
        # --- Condition 4: Naive LLM (Claude Opus) ---
        opus_map, opus_agg = run_naive_frontier_model(ctx, OPUS_MODEL_STRING)
        opus_geom_val = evaluate_mapping_operator(src_geom, opus_map)
        opus_sem_val = (opus_agg == gt_agg)
        
        # --- Condition 5: UrbanTrace Copilot (Live API) ---
        copilot_map, copilot_agg = run_copilot_api(ctx) 
        copilot_geom_val = evaluate_mapping_operator(src_geom, copilot_map)
        copilot_sem_val = (copilot_agg == gt_agg)
        
        # --- Compile Record ---
        record = {
            "scenario_id": f"{ctx['source_column']} -> {ctx['target_dataset']}",
            "context": ctx,
            "predictions": {
                "rule_based": {"geom_correct": bool(rb_geom_val), "sem_correct": bool(rb_sem_val), "map_out": rb_map, "agg_out": rb_agg},
                "naive_mini": {"geom_correct": bool(mini_geom_val), "sem_correct": bool(mini_sem_val), "map_out": mini_map, "agg_out": mini_agg},
                "naive_sonnet": {"geom_correct": bool(sonnet_geom_val), "sem_correct": bool(sonnet_sem_val), "map_out": sonnet_map, "agg_out": sonnet_agg},
                "naive_opus": {"geom_correct": bool(opus_geom_val), "sem_correct": bool(opus_sem_val), "map_out": opus_map, "agg_out": opus_agg},
                "copilot_api": {"geom_correct": bool(copilot_geom_val), "sem_correct": bool(copilot_sem_val), "map_out": copilot_map, "agg_out": copilot_agg}
            }
        }
        results.append(record)
        
    # --- Save Master JSON ---
    with open(MASTER_RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)
        
    # --- Generate Final Table ---
    summary_data = []
    for r in results:
        p = r["predictions"]
        summary_data.append({
            "RB_Geom": p["rule_based"]["geom_correct"], "RB_Sem": p["rule_based"]["sem_correct"],
            "Mini_Geom": p["naive_mini"]["geom_correct"], "Mini_Sem": p["naive_mini"]["sem_correct"],
            "Sonnet_Geom": p["naive_sonnet"]["geom_correct"], "Sonnet_Sem": p["naive_sonnet"]["sem_correct"],
            "Opus_Geom": p["naive_opus"]["geom_correct"], "Opus_Sem": p["naive_opus"]["sem_correct"],
            "Copilot_Geom": p["copilot_api"]["geom_correct"], "Copilot_Sem": p["copilot_api"]["sem_correct"],
        })
        
    df = pd.DataFrame(summary_data)
    
    metrics = {
        "Architecture Condition": [
            "Rule-Based (Legacy)", 
            "Naive LLM (GPT-4o-Mini)", 
            "Naive LLM (claude-sonnet)", 
            "Naive LLM (Claude Opus)", 
            "UrbanTrace Copilot (Claude Opus + Context)"
        ],
        "Geometric Validity (%)": [
            df["RB_Geom"].mean() * 100, 
            df["Mini_Geom"].mean() * 100, 
            df["Sonnet_Geom"].mean() * 100,
            df["Opus_Geom"].mean() * 100, 
            df["Copilot_Geom"].mean() * 100
        ],
        "Semantic Validity (%)": [
            df["RB_Sem"].mean() * 100, 
            df["Mini_Sem"].mean() * 100, 
            df["Sonnet_Sem"].mean() * 100,
            df["Opus_Sem"].mean() * 100, 
            df["Copilot_Sem"].mean() * 100
        ]
    }
    
    print(f"\n✅ Detailed trace log saved to '{MASTER_RESULTS_FILE}'")
    print("\n🎉 === FINAL MASTER EVALUATION COMPLETE ===")
    print(pd.DataFrame(metrics).to_string(index=False))

if __name__ == "__main__":
    run_master_benchmark()

🚀 Initializing Master Benchmark across 5 conditions (100 scenarios)...


Evaluating Scenarios:   0%|          | 0/100 [00:00<?, ?it/s]


✅ Detailed trace log saved to 'benchmark_results_master_no_geometry_from_benchmark.json'

🎉 === FINAL MASTER EVALUATION COMPLETE ===
                    Architecture Condition  Geometric Validity (%)  Semantic Validity (%)
                       Rule-Based (Legacy)                    33.0                   28.0
                   Naive LLM (GPT-4o-Mini)                     7.0                   51.0
                 Naive LLM (claude-sonnet)                    52.0                   80.0
                   Naive LLM (Claude Opus)                    74.0                   94.0
UrbanTrace Copilot (Claude Opus + Context)                    87.0                  100.0
